In [2]:
from sklearn.model_selection import StratifiedShuffleSplit
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from pylab import rcParams
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
import tensorflow as tf
import keras
from keras.models import Sequential, load_model
from keras.layers import Dropout, Dense, GlobalAveragePooling2D
from keras.optimizers import Adam
import efficientnet.keras as efn

In [7]:
train_data_dir = 'D:/Unicamp_D/INF-0619_Projeto-Final/data/OCT2017/train'
val_data_dir = 'D:/Unicamp_D/INF-0619_Projeto-Final/data/OCT2017/test'

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

In [57]:
batch_size = 16
img_rows, img_cols = 224, 224

train_generator = train_datagen.flow_from_directory(train_data_dir,
                                                    target_size=(img_rows, img_cols),
                                                    batch_size=batch_size,
                                                    class_mode='categorical')

Found 83484 images belonging to 4 classes.


In [58]:
validation_generator = validation_datagen.flow_from_directory(val_data_dir,
                                                              target_size=(img_rows, img_cols),
                                                              batch_size=batch_size,
                                                              class_mode='categorical')

Found 968 images belonging to 4 classes.


In [59]:
input_shape = (img_rows, img_cols, 3)

In [60]:
n_classes = 4
epochs = 15
batch_size = 16

In [61]:
model = efn.EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape, classes=n_classes)
model.summary()

for layer in model.layers:
    layer.trainable = False

model = tf.keras.Sequential([
                                    model,
                                    tf.keras.layers.GlobalAveragePooling2D(),
                                    tf.keras.layers.Dense(4, activation='softmax')
                                    ])

model.summary()

Model: "efficientnet-b0"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_7 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
stem_conv (Conv2D)              (None, 112, 112, 32) 864         input_7[0][0]                    
__________________________________________________________________________________________________
[... 271 linhas omitidas na versão pública ...]
block5b_activation (Activation) (None, 14, 14, 672)  0           block5b_bn[0][0]                 
__________________________________________________________________________________________________
block5b_se_squeeze (GlobalAvera (None, 672)          0           block5b_activation[0][0]         
____________________________________

block5b_se_expand (Conv2D)      (None, 1, 1, 672)    19488       block5b_se_reduce[0][0]          
__________________________________________________________________________________________________
block5b_se_excite (Multiply)    (None, 14, 14, 672)  0           block5b_activation[0][0]         
                                                                 block5b_se_expand[0][0]          
__________________________________________________________________________________________________
block5b_project_conv (Conv2D)   (None, 14, 14, 112)  75264       block5b_se_excite[0][0]          
__________________________________________________________________________________________________
block5b_project_bn (BatchNormal (None, 14, 14, 112)  448         block5b_project_conv[0][0]       
[... 190 linhas omitidas na versão pública ...]
top_bn (BatchNormalization)     (None, 7, 7, 1280)   5120        top_conv[0][0]                   
_____________________________________________________________

Model: "sequential_5"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
efficientnet-b0 (Functional) (None, 7, 7, 1280)        4049564   
_________________________________________________________________
global_average_pooling2d_5 ( (None, 1280)              0         
_________________________________________________________________
dense_5 (Dense)              (None, 4)                 5124      
Total params: 4,054,688
Trainable params: 5,124
Non-trainable params: 4,049,564
_________________________________________________________________


In [62]:
from keras.callbacks import ModelCheckpoint, EarlyStopping
checkpoint = ModelCheckpoint("base_model.h5",
                             monitor="val_loss",
                             mode="min",
                             save_best_only = True,
                             verbose=1)

earlystop = EarlyStopping(monitor = 'val_loss', 
                          min_delta = 0, 
                          patience = 3,
                          verbose = 1,
                          restore_best_weights = True)

callbacks = [earlystop, checkpoint]

In [63]:
model.compile(loss='categorical_crossentropy',
             optimizer=Adam(learning_rate=0.001),
             metrics=['accuracy'])

In [64]:
epochs = 10

nb_train_samples = 10000
nb_val_samples = 2000

In [65]:
history = model.fit(
    train_generator,
    steps_per_epoch = nb_train_samples // batch_size,
    epochs = epochs,
    callbacks = callbacks,
    validation_data = validation_generator,
    validation_steps = nb_val_samples // batch_size)

Epoch 1/10
625/625 [==============================] - ETA: 0s - loss: 0.6019 - accuracy: 0.7874WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 125 batches). You may need to use the repeat() function when building your dataset.

Epoch 00001: val_loss improved from inf to 0.49304, saving model to base_model.h5
625/625 [==============================] - 1017s 2s/step - loss: 0.6019 - accuracy: 0.7874 - val_loss: 0.4930 - val_accuracy: 0.7696
Epoch 2/10
625/625 [==============================] - 899s 1s/step - loss: 0.4563 - accuracy: 0.8381
Epoch 3/10
625/625 [==============================] - 897s 1s/step - loss: 0.3950 - accuracy: 0.8644
Epoch 4/10
625/625 [==============================] - 901s 1s/step - loss: 0.3761 - accuracy: 0.8694
Epoch 5/10
406/625 [==================>...........] - ETA: 3:55:02 - loss: 0.3668 - accuracy: 0.8717

KeyboardInterrupt: 

In [66]:
class_labels = validation_generator.class_indices
class_labels = {v: k for k, v in class_labels.items()}
classes = list(class_labels.values())

In [67]:
y_pred = model.predict(validation_generator, nb_val_samples // batch_size+1)

In [68]:
y_pred_label = np.argmax(y_pred, axis=1)

In [70]:
#Confusion Matrix and Classification Report
print('Confusion Matrix')
print(confusion_matrix(validation_generator.classes, y_pred_label))
print('Classification Report')
print(classification_report(validation_generator.classes, y_pred_label, target_names=classes))

Confusion Matrix
[[70 50 53 69]
 [76 53 55 58]
 [78 63 38 63]
 [84 50 41 67]]
Classification Report
              precision    recall  f1-score   support

         CNV       0.23      0.29      0.25       242
         DME       0.25      0.22      0.23       242
      DRUSEN       0.20      0.16      0.18       242
      NORMAL       0.26      0.28      0.27       242

    accuracy                           0.24       968
   macro avg       0.23      0.24      0.23       968
weighted avg       0.23      0.24      0.23       968

